# Fundamentals of Machine Learning

This demonstration notebook builds a small, reproducible machine-learning workflow and asks a trust question at each stage: **what evidence would make us trust this model's behavior?**

We will use a tabular classification problem to explore data validation, preprocessing, model comparison, fairness slices, interpretability, robustness, and an interactive prediction panel.

## 1. Notebook Scaffold

The notebook assumes it is launched from the repository root. All generated artifacts go into `artifacts/`, which can be created locally and ignored by Git.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
ARTIFACT_DIR = REPO_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
print(f"Repository root: {REPO_ROOT}")

## 2. Environment Setup and Imports

These dependencies support the workflow and the interactive controls. Version printouts make the computational environment part of the evidence.

In [ ]:
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print({"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__})
print("ipywidgets available:", importlib.util.find_spec("ipywidgets") is not None)

## 3. Reproducibility Controls

A trustworthy result should be repeatable. We fix the random seed and keep the main experimental choices in one configuration object.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    seed: int = 420
    test_size: float = 0.25
    cv_folds: int = 5
    target: str = "target"

CONFIG = Config()
np.random.seed(CONFIG.seed)


## 4. Dataset Loading and Validation

We use the breast-cancer dataset packaged with scikit-learn. The validation function makes schema, target, and missing-value assumptions visible before training.

In [ ]:
from sklearn.datasets import load_breast_cancer

def load_and_validate_dataset():
    raw = load_breast_cancer(as_frame=True)
    features = raw.data.copy()
    target = pd.Series(raw.target, name=CONFIG.target)
    if features.empty or target.empty:
        raise ValueError("Dataset must contain rows and feature columns.")
    if len(features) != len(target):
        raise ValueError("Features and target must have the same number of rows.")
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes):
        raise TypeError("This teaching example expects numeric feature columns.")
    if features.isna().all(axis=None):
        raise ValueError("All feature values are missing.")
    return features, target

X, y = load_and_validate_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG.test_size, random_state=CONFIG.seed, stratify=y
)
print("shape:", X.shape, "classes:", sorted(y.unique()), "missing values:", int(X.isna().sum().sum()))

## 5. Exploratory Data Analysis Functions

These functions make distributions, class balance, correlations, and missingness inspectable before we trust a model.

In [ ]:
def summarize_missingness(frame):
    return frame.isna().sum().sort_values(ascending=False).rename("missing").to_frame()

def plot_eda(features, target):
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.histplot(features.iloc[:, 0], kde=True, ax=axes[0])
    axes[0].set_title("Feature distribution")
    target.value_counts().sort_index().plot.bar(ax=axes[1], color=["#4c78a8", "#f58518"])
    axes[1].set_title("Class balance")
    sns.heatmap(features.iloc[:, :10].corr(), cmap="vlag", center=0, ax=axes[2])
    axes[2].set_title("Correlation: first 10 features")
    figure.tight_layout()
    return figure

plot_eda(X, y)
display(summarize_missingness(X).head())

## 6. Preprocessing Pipeline Construction

Pipelines prevent train/test leakage by fitting imputation, scaling, and encoding only on training data. This dataset is numeric, but the transformer also shows where categorical features would enter a mixed table.

In [ ]:
numeric_features = X.columns.tolist()
categorical_features = []

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

## 7. Baseline Model Training

A baseline gives us a reference point. Accuracy is reported alongside F1 and ROC-AUC so that one metric does not hide important behavior.

In [ ]:
def make_pipeline(model):
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

def evaluate(model, features, target):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]
    return {
        "accuracy": accuracy_score(target, predictions),
        "f1": f1_score(target, predictions),
        "roc_auc": roc_auc_score(target, probabilities),
    }

baseline = make_pipeline(DecisionTreeClassifier(max_depth=3, random_state=CONFIG.seed))
baseline.fit(X_train, y_train)
print(evaluate(baseline, X_test, y_test))
print(classification_report(y_test, baseline.predict(X_test)))

## 8. Model Comparison and Cross-Validation

Cross-validation compares candidates under the same folds. We select by mean ROC-AUC and retain the spread as a reminder that performance is uncertain.

In [ ]:
candidates = {
    "decision_tree": make_pipeline(DecisionTreeClassifier(max_depth=4, random_state=CONFIG.seed)),
    "random_forest": make_pipeline(RandomForestClassifier(n_estimators=150, random_state=CONFIG.seed, n_jobs=-1)),
}
folds = StratifiedKFold(n_splits=CONFIG.cv_folds, shuffle=True, random_state=CONFIG.seed)
comparison = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=folds, scoring="roc_auc")
    comparison.append({"model": name, "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
comparison = pd.DataFrame(comparison).sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
selected_name = comparison.loc[0, "model"]
selected_model = candidates[selected_name].fit(X_train, y_train)
display(comparison)
print("Selected:", selected_name)

## 9. Metric Computation and Visualization

In [ ]:
test_metrics = evaluate(selected_model, X_test, y_test)
predictions = selected_model.predict(X_test)
probabilities = selected_model.predict_proba(X_test)[:, 1]
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion matrix")
for label, color in [("positive", "#f58518"), ("negative", "#4c78a8")]:
    threshold = np.linspace(0.05, 0.95, 19)
    values = [f1_score(y_test, probabilities >= value) for value in threshold]
    axes[1].plot(threshold, values, label=label if label == "positive" else "F1", color=color)
axes[1].set(xlabel="Decision threshold", ylabel="F1", title="Threshold sensitivity")
axes[1].legend()
figure.tight_layout()
print(test_metrics)

## 10. Fairness and Slice-Based Error Analysis

The public dataset has no protected-group label, so this example uses a transparent proxy slice based on a feature median. It is a demonstration of the mechanics, not a claim about protected-group fairness.

In [ ]:
slice_feature = X_test.columns[0]
slice_labels = np.where(X_test[slice_feature] <= X_train[slice_feature].median(), "lower", "upper")
slice_rows = []
for slice_name in ["lower", "upper"]:
    mask = slice_labels == slice_name
    slice_rows.append({"slice": slice_name, "n": int(mask.sum()), **evaluate(selected_model, X_test.loc[mask], y_test.loc[mask])})
slice_metrics = pd.DataFrame(slice_rows)
slice_metrics["accuracy_gap_from_best"] = slice_metrics["accuracy"].max() - slice_metrics["accuracy"]
display(slice_metrics)


## 11. Interpretability Utilities

Permutation importance estimates which input features most affect held-out performance. It supports inspection, not causal claims.

In [ ]:
importance = permutation_importance(
    selected_model, X_test, y_test, n_repeats=5, random_state=CONFIG.seed, scoring="roc_auc"
)
importance_table = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)
display(importance_table.head(10))


## 12. Robustness and Stress Testing

We simulate two small deployment problems: missing values and measurement noise. Performance deltas make sensitivity easier to compare than a single score.

In [ ]:
stress_cases = {"clean": X_test.copy()}
noisy = X_test.copy()
noise = np.random.default_rng(CONFIG.seed).normal(0, 0.05, size=noisy.shape)
noisy = noisy + noise * X_train.std(axis=0).replace(0, 1).to_numpy()
stress_cases["measurement_noise"] = noisy
missing = X_test.copy()
missing.iloc[:, :3] = np.nan
stress_cases["missing_values"] = missing

robustness = []
for case_name, case_features in stress_cases.items():
    robustness.append({"case": case_name, **evaluate(selected_model, case_features, y_test)})
robustness = pd.DataFrame(robustness)
robustness["accuracy_delta"] = robustness["accuracy"] - robustness.loc[0, "accuracy"]
display(robustness)

## 13. Interactive UI with `ipywidgets`

Use the controls to change the candidate model, decision threshold, and held-out sample. The callback updates the prediction and metrics so students can see how choices change the evidence.

In [ ]:
def interactive_prediction(model_name, threshold, sample_index):
    model = candidates[model_name].fit(X_train, y_train)
    sample = X_test.iloc[[sample_index]]
    probability = model.predict_proba(sample)[0, 1]
    prediction = int(probability >= threshold)
    metrics = evaluate(model, X_test, y_test)
    print(f"Model: {model_name} | sample: {sample.index[0]}")
    print(f"Predicted class: {prediction} | probability of class 1: {probability:.3f}")
    print(f"True class: {int(y_test.iloc[sample_index])} | threshold: {threshold:.2f}")
    print("Held-out metrics:", {key: round(value, 3) for key, value in metrics.items()})

interact(
    interactive_prediction,
    model_name=Dropdown(options=list(candidates), value=selected_name, description="Model"),
    threshold=FloatSlider(min=0.10, max=0.90, step=0.05, value=0.50, description="Threshold"),
    sample_index=IntSlider(min=0, max=len(X_test) - 1, step=1, value=0, description="Sample"),
);

## 14. Artifact Saving and Reuse

Persisting the selected pipeline and its evaluation table makes the result auditable and reusable in a fresh session.

In [ ]:
import joblib

model_path = ARTIFACT_DIR / "selected_model.joblib"
metrics_path = ARTIFACT_DIR / "metrics.csv"
joblib.dump(selected_model, model_path)
pd.DataFrame([test_metrics]).to_csv(metrics_path, index=False)
print("Saved:", model_path, metrics_path)

# In a fresh session:
# loaded_model = joblib.load("artifacts/selected_model.joblib")
# loaded_model.predict(new_dataframe)